In [4]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import TimeSeriesSplit

In [5]:
# Project directories
PROJECT_ROOT = Path.cwd()

FEATURE_DIR = PROJECT_ROOT / "data" / "processed" / "features"
TARGET_DIR = PROJECT_ROOT / "data" / "processed" / "volatility_targets"
BASELINE_DIR = PROJECT_ROOT / "data" / "processed" / "baseline_predictions"

ML_OUTPUT_DIR = PROJECT_ROOT / "data" / "processed" / "ml_datasets"
ML_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Identify available stock files
feature_files = sorted(FEATURE_DIR.glob("*_features.parquet"))
target_files = sorted(TARGET_DIR.glob("*_targets.parquet"))

print("Feature files:", len(feature_files))
print("Target files:", len(target_files))
print("Feature example:", feature_files[0].name)
print("Target example:", target_files[0].name)
print("ML output directory:", ML_OUTPUT_DIR)

Feature files: 51
Target files: 50
Feature example: ADANIENT_NS_features.parquet
Target example: ADANIENT_NS_targets.parquet
ML output directory: /Users/srisuphachawla/Documents/DataScience_Projects/Indian_Equity_Market_Risk_Forecaster/notebooks/data/processed/ml_datasets


In [6]:
feature_names = {f.name for f in feature_files}
target_names = {f.name for f in target_files}

expected_feature_names = {
    f.name.replace("_targets.parquet", "_features.parquet")
    for f in target_files
}

print("Extra feature files:")
print(sorted(feature_names - expected_feature_names))

print("\nMissing feature files:")
print(sorted(expected_feature_names - feature_names))

Extra feature files:
['market_macro_features.parquet']

Missing feature files:
[]


In [7]:
# Assemble the combined ML dataset 

def load_with_date(path):
    df = pd.read_parquet(path)

    if "Date" not in df.columns:
        df = df.reset_index()
        if "Date" not in df.columns:
            df = df.rename(columns={df.columns[0]: "Date"})

    df["Date"] = pd.to_datetime(df["Date"])
    return df


# Load shared market/macro features
macro_features = load_with_date(
    FEATURE_DIR / "market_macro_features.parquet"
)

# Assemble stock-level datasets
stock_datasets = []

for feature_file in feature_files:
    if feature_file.name == "market_macro_features.parquet":
        continue

    ticker = feature_file.name.replace("_features.parquet", "")

    features = load_with_date(feature_file)
    targets = load_with_date(
        TARGET_DIR / f"{ticker}_targets.parquet"
    )

    stock_data = features.merge(
        targets,
        on="Date",
        how="inner",
        validate="one_to_one"
    )

    stock_data["Ticker"] = ticker
    stock_datasets.append(stock_data)


# Combine all stocks
ml_dataset = pd.concat(stock_datasets, ignore_index=True)

# Attach shared market/macro features
ml_dataset = ml_dataset.merge(
    macro_features,
    on="Date",
    how="left",
    validate="many_to_one"
)

ml_dataset = ml_dataset.sort_values(
    ["Ticker", "Date"]
).reset_index(drop=True)

# Save combined dataset
output_path = ML_OUTPUT_DIR / "combined_ml_dataset.parquet"
ml_dataset.to_parquet(output_path, index=False)

print("Dataset shape:", ml_dataset.shape)
print("Date range:", ml_dataset["Date"].min(), "to", ml_dataset["Date"].max())
print("Saved to:", output_path)

Dataset shape: (191857, 24)
Date range: 2010-01-04 00:00:00 to 2026-11-06 00:00:00
Saved to: /Users/srisuphachawla/Documents/DataScience_Projects/Indian_Equity_Market_Risk_Forecaster/notebooks/data/processed/ml_datasets/combined_ml_dataset.parquet


In [8]:
# Define predictors
stock_features = [
    "Return_Lag_1",
    "Return_Lag_2",
    "Return_Lag_5",
    "Return_Lag_10",
    "Return_Lag_21",
    "Hist_Vol_5D",
    "Hist_Vol_21D",
    "Hist_Vol_63D",
    "Volume_Change_1D",
    "Volume_MA_Ratio_5D",
    "Volume_MA_Ratio_21D",
    "Daily_Range"
]

market_features = [
    "NIFTY_Return",
    "NIFTY_Hist_Vol_21D",
    "INDIA_VIX_Level",
    "INDIA_VIX_Change",
    "USD_INR_Return",
    "BRENT_CRUDE_Return",
    "US_10Y_YIELD_Change"
]

feature_columns = stock_features + market_features

# Create separate datasets for each horizon
ml_5d = ml_dataset[
    ["Ticker", "Date"] + feature_columns + ["RV_5D"]
].copy()

ml_21d = ml_dataset[
    ["Ticker", "Date"] + feature_columns + ["RV_21D"]
].copy()

# Remove rows without valid targets
ml_5d = ml_5d.dropna(subset=["RV_5D"]).copy()
ml_21d = ml_21d.dropna(subset=["RV_21D"]).copy()

# Create log-volatility targets
ml_5d["Target"] = np.log(ml_5d["RV_5D"])
ml_21d["Target"] = np.log(ml_21d["RV_21D"])

# Save horizon-specific datasets
ml_5d.to_parquet(
    ML_OUTPUT_DIR / "ml_dataset_5D.parquet",
    index=False
)

ml_21d.to_parquet(
    ML_OUTPUT_DIR / "ml_dataset_21D.parquet",
    index=False
)

In [9]:
fold_records = {
    "5D": [
        ("2010-01-04", "2021-09-28", "2021-10-06", "2022-04-04"),
        ("2010-01-04", "2022-03-28", "2022-04-05", "2022-09-30"),
        ("2010-01-04", "2022-09-23", "2022-10-03", "2023-03-29"),
        ("2010-01-04", "2023-03-22", "2023-03-31", "2023-09-27"),
        ("2010-01-04", "2023-09-20", "2023-09-28", "2024-03-28"),
    ],
    "21D": [
        ("2010-01-04", "2021-09-03", "2021-10-06", "2022-03-30"),
        ("2010-01-04", "2022-02-25", "2022-03-31", "2022-09-22"),
        ("2010-01-04", "2022-08-23", "2022-09-23", "2023-03-15"),
        ("2010-01-04", "2023-02-13", "2023-03-16", "2023-09-07"),
        ("2010-01-04", "2023-08-08", "2023-09-08", "2024-03-04"),
    ]
}

fold_rows = []

for horizon, records in fold_records.items():
    for fold_number, (train_start, train_end, val_start, val_end) in enumerate(records, 1):
        fold_rows.append({
            "Horizon": horizon,
            "Fold": fold_number,
            "Train_Start": pd.Timestamp(train_start),
            "Train_End": pd.Timestamp(train_end),
            "Validation_Start": pd.Timestamp(val_start),
            "Validation_End": pd.Timestamp(val_end)
        })

fold_results = pd.DataFrame(fold_rows)

fold_results.to_csv(
    ML_OUTPUT_DIR / "walk_forward_folds.csv",
    index=False
)

display(fold_results)

,Horizon,Fold,Train_Start,Train_End,Validation_Start,Validation_End
0,5D,1,2010-01-04,2021-09-28,2021-10-06,2022-04-04
1,5D,2,2010-01-04,2022-03-28,2022-04-05,2022-09-30
2,5D,3,2010-01-04,2022-09-23,2022-10-03,2023-03-29
3,5D,4,2010-01-04,2023-03-22,2023-03-31,2023-09-27
4,5D,5,2010-01-04,2023-09-20,2023-09-28,2024-03-28
5,21D,1,2010-01-04,2021-09-03,2021-10-06,2022-03-30
6,21D,2,2010-01-04,2022-02-25,2022-03-31,2022-09-22
7,21D,3,2010-01-04,2022-08-23,2022-09-23,2023-03-15
8,21D,4,2010-01-04,2023-02-13,2023-03-16,2023-09-07
9,21D,5,2010-01-04,2023-08-08,2023-09-08,2024-03-04


### Note
We are going to add two levels of Validation. 

Outer folds: The five existing folds evaluate model performance and compare against baselines.

Inner folds: Time-ordered splits inside each outer training period select hyperparameters.

In [10]:
INNER_SPLITS = 3

inner_fold_results = {}

for horizon in ["5D", "21D"]:
    horizon_data = ml_5d if horizon == "5D" else ml_21d

    inner_fold_results[horizon] = {}

    for _, fold in fold_results[
        fold_results["Horizon"] == horizon
    ].iterrows():

        train_data = horizon_data[
            (horizon_data["Date"] >= fold["Train_Start"])
            & (horizon_data["Date"] <= fold["Train_End"])
        ]

        unique_dates = np.sort(train_data["Date"].unique())

        gap = 5 if horizon == "5D" else 21

        splitter = TimeSeriesSplit(
            n_splits=INNER_SPLITS,
            gap=gap
        )

        splits = []

        for inner_fold, (train_idx, val_idx) in enumerate(
            splitter.split(unique_dates), 1
        ):
            splits.append({
                "Inner_Fold": inner_fold,
                "Train_Start": unique_dates[train_idx[0]],
                "Train_End": unique_dates[train_idx[-1]],
                "Validation_Start": unique_dates[val_idx[0]],
                "Validation_End": unique_dates[val_idx[-1]],
                "Train_Dates": unique_dates[train_idx],
                "Validation_Dates": unique_dates[val_idx]
            })

        inner_fold_results[horizon][int(fold["Fold"])] = splits

print("Inner tuning splits created.")

for horizon in inner_fold_results:
    print(f"\n{horizon}")
    for fold_number, splits in inner_fold_results[horizon].items():
        print(f"Outer fold {fold_number}: {len(splits)} inner splits")

Inner tuning splits created.

5D
Outer fold 1: 3 inner splits
Outer fold 2: 3 inner splits
Outer fold 3: 3 inner splits
Outer fold 4: 3 inner splits
Outer fold 5: 3 inner splits

21D
Outer fold 1: 3 inner splits
Outer fold 2: 3 inner splits
Outer fold 3: 3 inner splits
Outer fold 4: 3 inner splits
Outer fold 5: 3 inner splits
